In [15]:
import os

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
import astropy.units as u
import statmorph

import utils.data as datutils
import utils.image as imutils
import utils.file as futils
import utils.plots as plots

In [16]:
def morph(map_dir, annulus=False, r1=1*u.Mpc, r2=50*u.kpc):
    finished_ids = []
    morphs_list = []
    files_list = sorted(os.listdir(map_dir))
    for file in files_list:
        id = futils.find_id(file)
        if id in finished_ids:
            continue
        else:
            print(f"Processing region {id}")
            finished_ids.append(id)

        map = futils.load_map(file, map_dir)
        if map is None:
            print(f"Skipping region {id}: no map")
            continue

        pixr1 = datutils.real2pix(r1, map)
        pixr2 = datutils.real2pix(r2, map)
        center = (int(len(map[1])//2), int(len(map[0])//2))
        if annulus:
            segmap = imutils.annular_mask(map, center, pixr2, pixr1)
        else:
            segmap = imutils.circular_segmap(map, center, pixr1)

        # fig, axs = plt.subplots(1, 1)
        # plots.display_img(map, axs, mask=segmap)
        # plt.show()
        morph = statmorph.source_morphology(map, segmap, gain=2.25)
        morphs_list.append((id, morph[0]))
        print(f'{file} done')

        if len(morphs_list) % 20 == 0:
            if annulus:
                rad2 = r2.to(u.kpc).value
                rad1 = r1.to(u.Mpc).value
                name = f'results/gadgetx3k_{len(morphs_list)}_rin{rad2}kpc_rout{rad1}Mpc.csv'
                sm_df = futils.create_morph_df(morphs_list,
                                                 name=name,
                                                 save=True)
            else:
                rad1 = r1.to(u.Mpc).value
                name = f'results/gadgetx3k_{len(morphs_list)}_{rad1}Mpc.csv'
                sm_df = futils.create_morph_df(morphs_list, 
                                                 name=name, 
                                                 save=True)

    if annulus:
        name = f'results/gadgetx3k_{len(morphs_list)}_rin{rad2}kpc_rout{rad1}Mpc_.csv'
        sm_df = futils.create_morph_df(morphs_list,
                                            name=name,
                                            save=True)
    else:
        name = f'results/gadgetx3k_{len(morphs_list)}_{rad1}Mpc_.csv'
        sm_df = futils.create_morph_df(morphs_list,
                                            name=name,
                                            save=True)

    return

In [17]:
morph('data/gadgetx3k_20/maps/OLD_ICs/', annulus=False, r1=1*u.Mpc, r2=60*u.kpc)

Processing region 1


KeyboardInterrupt: 